# Project 04 -- Volatility Modeling

GARCH-family and regime-switching models for conditional volatility estimation, forecasting, and risk measurement.

**Outline:**
1. Load SPY returns
2. Fit GARCH, GJR-GARCH, EGARCH -- model comparison table
3. Plot conditional volatility for best model
4. Standardized residual diagnostics (QQ plot, Ljung-Box)
5. Markov regime-switching model and regime probability plot
6. Conditional VaR and ES
7. Compare conditional VaR vs constant-volatility VaR from P01

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Ensure project src/ is importable
_PROJECT_SRC = str(Path.cwd().parent / "src")
if _PROJECT_SRC not in sys.path:
    sys.path.insert(0, _PROJECT_SRC)

# Shared library
from diagnostics import (
    ljung_box_test,
    plot_conditional_volatility,
    plot_model_comparison,
    plot_regime_probabilities,
    plot_standardized_residuals,
    plot_volatility_term_structure,
)

# Project-local
from model import VolatilityModel

from risk_analyst.data.market import compute_returns, fetch_prices
from risk_analyst.measures.var import expected_shortfall, historical_var, parametric_var
from risk_analyst.models.regime import (
    fit_regime_switching,
    regime_summary,
)
from risk_analyst.models.volatility import (
    conditional_es,
    conditional_var,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

print("Imports OK.")

## 1. Load SPY Returns

In [ ]:
# Load config
config_path = Path.cwd().parent / "configs" / "default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Config loaded:")
config

In [ ]:
# Fetch SPY prices and compute log returns
prices = fetch_prices(
    tickers=[config["data"]["ticker"]],
    start_date=config["data"]["start_date"],
    end_date=config["data"]["end_date"],
)

returns_df = compute_returns(prices, method=config["data"]["returns_type"])
returns = returns_df.iloc[:, 0]  # single-asset Series

print(f"SPY log returns: {len(returns)} observations, {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Mean: {returns.mean():.6f}, Std: {returns.std():.6f}, Skew: {returns.skew():.3f}, Kurt: {returns.kurtosis():.3f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(returns.index, returns.values, linewidth=0.4, color="steelblue")
ax.set_title("SPY Daily Log Returns")
ax.set_ylabel("Return")
plt.show()

## 2. Fit GARCH Variants and Compare

In [ ]:
# Instantiate VolatilityModel and fit all three
vm = VolatilityModel(config)
fitted = vm.fit_all(returns)

print("Fitted models:", list(fitted.keys()))

In [ ]:
# Model comparison table
comparison = vm.compare_models(returns)
print("Model Comparison (sorted by AIC):")
comparison

In [ ]:
# Bar chart comparison
fig = plot_model_comparison(comparison)
plt.show()

In [ ]:
# Identify best model
best_model_name = comparison.index[0]
best_model = fitted[best_model_name]
print(f"Best model by AIC: {best_model_name}")
print(best_model.summary())

## 3. Conditional Volatility -- Best Model

In [ ]:
fig = plot_conditional_volatility(
    returns, best_model,
    title=f"SPY Returns with Conditional Volatility ({best_model_name.upper()})",
)
plt.show()

## 4. Standardized Residual Diagnostics

In [ ]:
# QQ plot and histogram
fig = plot_standardized_residuals(best_model)
plt.show()

In [ ]:
# Ljung-Box test on squared standardized residuals
raw_resid = best_model.std_resid
if hasattr(raw_resid, "dropna"):
    std_resid = raw_resid.dropna().values
else:
    std_resid = np.asarray(raw_resid, dtype=np.float64)
    std_resid = std_resid[~np.isnan(std_resid)]

lb = ljung_box_test(std_resid, lags=config["diagnostics"]["ljung_box_lags"])
print("Ljung-Box Test on z_t^2 (squared standardized residuals):")
lb

If all p-values are above 0.05, the GARCH model has adequately captured the volatility clustering.

## 5. Regime-Switching Model

In [ ]:
# Fit Markov regime-switching
rs_cfg = config["models"]["regime_switching"]
regime_model = fit_regime_switching(
    returns,
    n_regimes=rs_cfg["n_regimes"],
    switching_variance=rs_cfg["switching_variance"],
)

summary = regime_summary(regime_model)
print("Regime means:", [f"{m:.6f}" for m in summary["means"]])
print("Regime variances:", [f"{v:.8f}" for v in summary["variances"]])
print("Regime std devs:", [f"{np.sqrt(v):.6f}" for v in summary["variances"]])
print("\nTransition matrix:")
print(summary["transition_matrix"])

In [ ]:
# Regime probabilities plot
fig = plot_regime_probabilities(returns, regime_model)
plt.show()

## 6. Conditional VaR and ES

In [ ]:
# Compute conditional risk measures at multiple confidence levels
results = []
for model_name in ["garch", "gjr_garch", "egarch"]:
    for alpha in config["risk"]["confidence_levels"]:
        risk = vm.compute_conditional_risk(model_name, alpha)
        results.append({
            "Model": model_name,
            "Alpha": alpha,
            "VaR": risk["VaR"],
            "ES": risk["ES"],
        })

risk_df = pd.DataFrame(results)
print("Conditional Risk Measures (1-day, decimal returns):")
risk_df

In [ ]:
# FHS-based VaR/ES for the best model
print(f"\nFiltered Historical Simulation ({best_model_name}):")
for alpha in config["risk"]["confidence_levels"]:
    fhs = vm.filtered_historical_simulation(returns, best_model_name, alpha, n_sims=50_000)
    print(f"  alpha={alpha}: VaR={fhs['VaR']:.6f}, ES={fhs['ES']:.6f}")

## 6b. Volatility Term Structure

In [ ]:
fig = plot_volatility_term_structure(best_model, horizons=list(range(1, 21)))
plt.show()

## 7. Conditional VaR vs Constant-Volatility VaR (P01 Comparison)

Constant-volatility VaR uses the *unconditional* (sample) standard deviation, ignoring time-varying volatility. Conditional VaR from GARCH adapts to the current regime.

In [ ]:
# Constant-vol VaR from P01 methods
losses = -returns.values  # positive = loss

comparison_rows = []
for alpha in config["risk"]["confidence_levels"]:
    hist_var = historical_var(losses, alpha)
    param_var = parametric_var(losses, alpha)
    cond_var = conditional_var(best_model, alpha)
    cond_es_val = conditional_es(best_model, alpha)
    hist_es = expected_shortfall(losses, alpha)

    comparison_rows.append({
        "Alpha": alpha,
        "Historical VaR (P01)": hist_var,
        "Parametric VaR (P01)": param_var,
        f"Conditional VaR ({best_model_name})": cond_var,
        "Historical ES (P01)": hist_es,
        f"Conditional ES ({best_model_name})": cond_es_val,
    })

comp_df = pd.DataFrame(comparison_rows).set_index("Alpha")
print("VaR Comparison: Constant-Vol (P01) vs. Conditional (P04)")
comp_df

In [ ]:
# Visual: rolling conditional VaR vs constant VaR
alpha_plot = 0.99

# Constant VaR (horizontal line)
const_var = parametric_var(losses, alpha_plot)

# Rolling conditional VaR: use conditional volatility time series
cond_vol_pct = best_model.conditional_volatility
cond_vol = np.asarray(cond_vol_pct) / 100.0  # decimal sigma

# Approximate VaR_t ~ sigma_t * |quantile|
from scipy import stats as sp_stats

if "nu" in best_model.params:
    nu = best_model.params["nu"]
    q = abs(sp_stats.t.ppf(1 - alpha_plot, df=nu))
else:
    q = abs(sp_stats.norm.ppf(1 - alpha_plot))

rolling_cond_var = cond_vol * q  # time-varying VaR

n = min(len(returns), len(rolling_cond_var))
idx = returns.index[-n:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(idx, np.abs(returns.values[-n:]), color="steelblue", linewidth=0.3, alpha=0.5, label="|Return|")
ax.plot(idx, rolling_cond_var[-n:], color="red", linewidth=0.8, label=f"Conditional VaR ({alpha_plot})")
ax.axhline(const_var, color="orange", linestyle="--", linewidth=1.0, label=f"Constant VaR ({alpha_plot})")
ax.set_title(f"Conditional vs. Constant VaR at {alpha_plot*100:.0f}%")
ax.set_ylabel("Loss / VaR")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Key takeaway:** Conditional VaR adapts to current market conditions -- it rises during turbulent periods (e.g., COVID-19 March 2020) and falls during calm periods. Constant-volatility VaR is flat and either over- or under-estimates risk depending on the regime.